# Thai Math VQA — Colab Pro pipeline (env → submission)

Open-weights VLM (no API), self-consistency voting, exact-match normalization.

**Runtime:** set `Runtime → Change runtime type → GPU` (A100 40GB or L4 24GB on Colab Pro).

Steps: GPU check → install → clone code → Kaggle data → sanity → smoke test → eval on train → predict → submit.

Default model `Qwen/Qwen2.5-VL-7B-Instruct` fits Colab Pro. For 32B you need A100 80GB or an AWQ quant.

## 1. Check the GPU

In [ ]:
!nvidia-smi


## 2. Install dependencies
vLLM pins a compatible torch. If Colab asks to **restart the runtime** after this, do it, then continue from cell 3 (do NOT re-run this cell).

In [ ]:
%pip install -q -U vllm "transformers>=4.49.0" qwen-vl-utils accelerate pythainlp kaggle pillow pandas
print('installed — if prompted, restart runtime then skip to cell 3')


## 3. Get the code
Clones your repo. (Push the pipeline first: `git push origin main`.)
Alternatively, upload the folder to `/content/thai-math-vqa` and skip the clone.

In [ ]:
import os
REPO = 'https://github.com/mingrath/thai-math-vqa.git'
if not os.path.isdir('/content/thai-math-vqa'):
    !git clone $REPO /content/thai-math-vqa
%cd /content/thai-math-vqa
!git pull --ff-only || true
!ls


## 4. Kaggle credentials
Get `kaggle.json` from Kaggle → Account → *Create New API Token*, then upload it here.

In [ ]:
from google.colab import files
import os
up = files.upload()  # choose kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json 2>/dev/null || cp $(ls *kaggle*.json | head -1) /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('kaggle creds set')


## 5. Download + unzip the dataset
Requires having accepted the competition rules on the Kaggle website first.

In [ ]:
SLUG = 'super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen'
!kaggle competitions download -c $SLUG -p data/
!cd data && unzip -q -o '*.zip' && rm -f *.zip && cd ..
!find data -maxdepth 2 -type d; echo '---'; ls data


## 6. Sanity-check data + normalization

In [ ]:
!python -m src.data data
print('--- normalize self-test ---')
!python src/normalize.py


## 7. Smoke test (16 train images, greedy)
First run downloads the model weights (~16GB for 7B) — give it a few minutes.

In [ ]:
!python run_pipeline.py --mode eval --limit 16 --n 1


## 8. Full eval on train (measure expected score)
Self-consistency n=8. This is your offline estimate of the leaderboard score and
shows sample misses to guide prompt/normalize tuning. Tune here before spending a submission.

In [ ]:
!python run_pipeline.py --mode eval --n 8 --save-eval eval_train.csv


## 9. Predict the test set → submission.csv
420 images x n samples. Writes `submissions/sub.csv` with columns `id,answer`.

In [ ]:
!python run_pipeline.py --mode predict --n 8 --out submissions/sub.csv
import pandas as pd; print(pd.read_csv('submissions/sub.csv').head(10)); print('rows:', len(pd.read_csv('submissions/sub.csv')))


## 10. Submit to Kaggle
Limit: 5 submissions total — submit your best eval config.

In [ ]:
SLUG = 'super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen'
!kaggle competitions submit -c $SLUG -f submissions/sub.csv -m 'qwen2.5-vl-7b self-consistency n=8'


## Tuning levers (raise the score)
- **More samples:** `--n 16` (compute is cheap; only 420 images). Biggest single lever.
- **Bigger / better model:** `--model Qwen/Qwen3-VL-8B-Instruct` (stronger Thai+math, needs recent vllm). On A100 80GB try `Qwen/Qwen2.5-VL-72B-Instruct-AWQ` with `--tp 1 --max-pixels 802816`.
- **Resolution:** `--max-pixels 1605632` (=2048*28*28) helps OCR-heavy Thai images if VRAM allows; lower it if you OOM.
- **Prompt:** edit `src/prompts.py`. Add 1–2 few-shot examples per answer format to nail exact-match shape.
- **Normalization misses:** inspect `eval_train.csv`; extend the Thai units list / rules in `src/normalize.py`.
- **Ensemble:** run predict with 2–3 *different* model families, then majority-vote across their `sub.csv` files (`src.normalize.majority_vote`).